Title: ST-554 Homework 10 \
Author: Stephen Griggs: \
Date: 4/20/2026

# Introduction
This notebook explores Spark Structured Streaming in two parts:

1. **Streaming with the `rate` source:**
    - Generate a stream using the built-in `rate` format.
    - Apply transformations to the `value` column using `pyspark.sql.functions`:
        - Square root of `value`
        - `value` mod 4
    - Write the results to an in-memory table via `writeStream`.
    - Let the query run briefly, then inspect the accumulated output.

2. **Streaming CSV files through a fitted Pipeline:**
    - Read `bikeDetails_for_fit.csv` into a Spark DataFrame.
    - Build and fit a Pipeline with two stages:
        - `SQLTransformer` for log transforms and a one-owner dummy variable.
        - `VectorAssembler` to assemble the features column.
    - Set up a file-based `readStream` to watch a folder for incoming `bikeDetails_add*.csv` files.
    - Transform each new file using the fitted pipeline and append the results to the console.

## Part 1: Streaming with the `rate` Source

First, create a streaming DataFrame using the `rate` format, apply `sqrt` and `mod 4` transformations to the `value` column, then write the results to an in-memory table. After letting the query run for ~30 seconds, it's stopped and the accumulated output is displayed.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, col
import time

spark = SparkSession.builder.appName("HW10_Streaming").getOrCreate()

# Quiet down Spark's WARN messages and streaming checkpoint/AQE warnings for cleaner output
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
spark.conf.set("spark.sql.adaptive.enabled", "false")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 13:32:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Set up the rate stream
rate_stream = spark.readStream.format("rate").load()

# Apply transformations: sqrt(value) and value mod 4
transformed = rate_stream.select(
    col("timestamp"),
    col("value"),
    sqrt(col("value")).alias("sqrt_value"),
    (col("value") % 4).alias("mod4_value")
)

# Write to memory and start the query
query = (transformed.writeStream
         .format("memory")
         .queryName("rate_table")
         .start())

# Let it run ~30 seconds, then stop
time.sleep(30)
query.stop()

# Show the collected results
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-20 13:33:...|    0|               0.0|         0|
|2026-04-20 13:33:...|    1|               1.0|         1|
|2026-04-20 13:33:...|    2|1.4142135623730951|         2|
|2026-04-20 13:33:...|    3|1.7320508075688772|         3|
|2026-04-20 13:33:...|    4|               2.0|         0|
|2026-04-20 13:33:...|    5|  2.23606797749979|         1|
|2026-04-20 13:33:...|    6| 2.449489742783178|         2|
|2026-04-20 13:33:...|    7|2.6457513110645907|         3|
|2026-04-20 13:33:...|    8|2.8284271247461903|         0|
|2026-04-20 13:33:...|    9|               3.0|         1|
|2026-04-20 13:33:...|   10|3.1622776601683795|         2|
|2026-04-20 13:33:...|   11|   3.3166247903554|         3|
|2026-04-20 13:33:...|   12|3.4641016151377544|         0|
|2026-04-20 13:33:...|   13| 3.605551275463989|         

## Part 2: Streaming CSV Files Through a Fitted Pipeline

Read `bikeDetails_for_fit.csv` into a Spark DataFrame, then build and fit a two-stage Pipeline:

1. `SQLTransformer`: log-transforms selling_price as a label and km_driven, then creates a one_owner dummy.
2. `VectorAssembler`: assembles year, log_km_driven, and one_owner into a features column.

Then set up a file-based readStream watching a folder for the `bikeDetails_add*.csv` files, apply the fitted pipeline's .transform() to each incoming file, and append the results to the console.

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler

# Read the fit CSV as a Spark DataFrame
bike_fit = spark.read.csv("./bikeDetails_for_fit.csv", header=True, inferSchema=True)

# Stage 1: SQL transform (log transforms, rename, dummy variable)
sql_trans = SQLTransformer(statement="""
    SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
           CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
    FROM __THIS__
""")

# Stage 2: Vector assembler for features
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"],
    outputCol="features"
)

# Build and fit the pipeline
pipeline = Pipeline(stages=[sql_trans, assembler])
fitted_pipeline = pipeline.fit(bike_fit)

Set up a file-based `readStream` on the `stream-data/` folder, then copy the five CSV files in one at a time, apply the fitted pipeline to each incoming file, and write results to the console in append mode.

In [4]:
import os, shutil, glob

# Ensure the streaming folder exists and is empty
stream_dir = "stream-data"
os.makedirs(stream_dir, exist_ok=True)
for f in glob.glob(f"{stream_dir}/*.csv"):
    os.remove(f)

# Read stream watching the folder for incoming files
bike_stream = (spark.readStream
               .schema(bike_fit.schema)
               .option("header", True)
               .csv(stream_dir))

# Transform each incoming file with the fitted pipeline
transformed_stream = fitted_pipeline.transform(bike_stream)

# Write to console in append mode
stream_query = (transformed_stream.writeStream
                .format("console")
                .outputMode("append")
                .option("numRows", 10)
                .option("truncate", False)
                .start())

# Move the five files into the folder one at a time
source_files = sorted(glob.glob("bikeDetails_add*.csv"))
for src in source_files:
    time.sleep(3)
    shutil.copy(src, os.path.join(stream_dir, os.path.basename(src)))
    print(f"Added {os.path.basename(src)}")

# Let the last file finish processing, then stop
time.sleep(5)
stream_query.stop()

Added bikeDetails_add1.csv
-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|8.987196820661973 |2003|10.887436932884098|1        |[2003.0,10.887436932884098,1.0]|
|11.156250521031495|2018|9.615805480084347 |1        |[2018.0,9.615805480084347,1.0] |
|10.819778284410283|2016|8.987196820661973 |1        |[2016.0,8.987196820661973,1.0] |
|10.46310334047155 |2015|10.582738627903963|1        |[2015.0,10.582738627903963,1.0]|
|9.903487552536127 |2006|11.225243392518447|1        |[2006.0,11.225243392518447,1.0]|
|10.819778284410283|2012|10.239959789157341|1        |[2012.0,10.239959789157341,1.0]|
|10.51867319162636 |2008|11.03488966402723 |1        |[2008.0,11.03488966402723,1.0] |
|11.14